# Task 2 — Moving an object through time
## Demonstration process and high-level acceptance script

**Purpose:** work through the required behaviour together while implementing Task 2 in parallel with Task 1. This is a demonstration scaffold, not a completed temporal engine.

**Based on:** the supplied assignment, §§2.1.1, 2.1.2 and 2.2, and the starter's `applicable(wm, action, t)` / `step(wm, action, t)` interfaces. Backward planning (§2.3) is optional and excluded here.

For each section: **predict → explain → run → compare**. Rotate driver, predictor and reviewer. The knowledge and expected states below run immediately using only Python's standard library. Engine checks are visibly **SKIPPED** until enabled; a skipped check is not a pass.

Unlike Task 1, do not keep firing all applicable actions until a fixpoint. Select **one action per time step**. Actions cause a new state; they do not merely infer more facts about the same state.

## 1. Agree the representation before implementing

| Item | Contract used in this notebook |
|---|---|
| Stored fact | Ground tuple; integer time is the last argument, e.g. `("at", "obj1", "A", 0)` |
| Conditions | Timed patterns, all checked at the requested `t`; bind `?t` before matching |
| Action schema | Dictionary with `name`, `if`, `add`, `delete` |
| Effects | Grounded **untimed** tuples; engine stamps them with `t+1` |
| Applicable instance | Dictionary with grounded `name`, `add`, `delete`; optionally other fields |
| `step` input here | One grounded applicable instance, not an unbound schema |
| Deletion | Prevents a fact from being carried into the next state; history remains |
| Progress | Exactly one action and one advance from `t` to `t+1` per call |

The starter's comments alternate between `terminate` and `delete`. This notebook consistently uses **`delete`**, matching its example action dictionary. The starter permits you to choose the shape of grounded instances; if yours differs, adapt only the adapter cell below.

Use a fresh branch of working memory when replaying a scenario. Do not append two alternative outcomes at the same time step into one history.

A held object's position is represented by `holding(obj1, t)` together with the hand's location. It has no separate `at(obj1, ..., t)` while held. This is a domain choice, not an engine special case.

In [ ]:
# Knowledge fixture: ultimately place this data in domains.py.
DEMO_FACTS = {
    ("at", "hand", "A", 0),
    ("at", "obj1", "A", 0),
    ("at", "obj2", "C", 0),
    ("empty", "hand", 0),
    ("object", "obj1", 0), ("object", "obj2", 0),
    ("location", "A", 0), ("location", "B", 0), ("location", "C", 0),
    # Legal directed moves; explicitly include reverse links if needed.
    ("route", "A", "B", 0), ("route", "B", "C", 0),
}

ACTIONS = [
    {
        "name": ("grab", "?o", "?loc"),
        "if": [("object", "?o", "?t"),
               ("at", "hand", "?loc", "?t"),
               ("at", "?o", "?loc", "?t"), ("empty", "hand", "?t")],
        "add": [("holding", "?o")],
        "delete": [("empty", "hand"), ("at", "?o", "?loc")],
    },
    {
        "name": ("move", "?from", "?to"),
        "if": [("at", "hand", "?from", "?t"),
               ("route", "?from", "?to", "?t"),
               ("location", "?to", "?t")],
        "add": [("at", "hand", "?to")],
        "delete": [("at", "hand", "?from")],
    },
    {
        "name": ("release", "?o", "?loc"),
        "if": [("object", "?o", "?t"), ("holding", "?o", "?t"),
               ("at", "hand", "?loc", "?t")],
        "add": [("at", "?o", "?loc"), ("empty", "hand")],
        "delete": [("holding", "?o")],
    },
]
SCHEMAS = {action["name"][0]: action for action in ACTIONS}

# Expected grounded instance, written by hand, not derived by a reasoner.
EXPECTED_GRAB = {
    "name": ("grab", "obj1", "A"),
    "add": [("holding", "obj1")],
    "delete": [("empty", "hand"), ("at", "obj1", "A")],
}
print("Initial facts:", len(DEMO_FACTS), "| Action schemas:", len(ACTIONS))

**Discuss:** why is `object(?o, ?t)` in `grab`? Without it, the hand could match the object variable and the system might grab itself. This restriction belongs in action data. Why is `move` valid whether or not the hand is empty? Its effects change the hand's position; `holding` persists without being listed in the action.

Locations, object declarations and routes also carry time and must persist. The engine must not treat these predicates as timeless exceptions.

In [ ]:
# Display/check helpers only: these are not the temporal engine.
def at_time(facts, t):
    return {fact for fact in facts if fact[-1] == t}

def contents(facts, t):
    """Untimed view for comparing two states without their timestamps."""
    return {fact[:-1] for fact in at_time(facts, t)}

def show_state(facts, t):
    state = at_time(facts, t)
    print(f"\nSTATE t={t} ({len(state)} facts)")
    for fact in sorted(state, key=repr):
        print(" ", fact)

def check_state(facts, t, expected):
    actual = at_time(facts, t)
    assert actual == expected, {
        "missing": expected - actual, "unexpected": actual - expected
    }
    print(f"PASS: exact state at t={t}")

show_state(DEMO_FACTS, 0)

## 2. Connect your engine when ready

Default mode is a readable walkthrough. Set `RUN_ENGINE_CHECKS = True` after implementing `applicable` and `step`. Change `ENGINE_MODULE` to your import path; `reasoning.engine` matches the structure discussed. Install your project into the notebook's environment if imports fail (for example, `pip install -e .` from the repository root).

The notebook does not implement either function. When enabled, missing functions, errors and failed assertions are exposed rather than silently treated as successful tests. Restart the kernel and run all cells after changing engine code.

In [ ]:
RUN_ENGINE_CHECKS = False
ENGINE_MODULE = "reasoning.engine"  # Or "engine" for a flat project.

if RUN_ENGINE_CHECKS:
    import importlib
    engine = importlib.import_module(ENGINE_MODULE)
    WorkingMemory = engine.WorkingMemory
    print("Engine loaded:", ENGINE_MODULE)
else:
    print("WALKTHROUGH MODE: engine acceptance checks will be SKIPPED.")

# Adapt these two wrappers if your agreed instance shape differs.
def instances(wm, schema, t):
    return list(engine.applicable(wm, schema, t))

def apply_instance(wm, instance, t):
    # Contract: step mutates WM and returns that updated WM, as in the starter.
    result = engine.step(wm, instance, t)
    assert result is wm, "Adapt this wrapper if step returns a fresh WM."
    return result

def choose(wm, name, t):
    options = instances(wm, SCHEMAS[name[0]], t)
    matching = [a for a in options if tuple(a["name"]) == name]
    assert len(matching) == 1, ("Expected one applicable instance", name, matching)
    return matching[0]

def acceptance(label, test):
    if not RUN_ENGINE_CHECKS:
        print("SKIPPED:", label)
        return
    test()
    print("PASS:", label)

## 3. Predict applicability at time 0

Expected action instances: **grab obj1 at A**, and **move A → B**. No release is available. `obj2` is at C, so it cannot be grabbed at A. Matching must carry the same `?loc` through all conditions.

Applicability should only read WM; it must not add effects, delete facts or advance time. Materialise the generator before changing WM to avoid iterating over a set while modifying it.

In [ ]:
def test_initial_applicability():
    wm = WorkingMemory(DEMO_FACTS)
    before = wm.snapshot()
    names = {tuple(a["name"]) for schema in ACTIONS for a in instances(wm, schema, 0)}
    assert names == {("grab", "obj1", "A"), ("move", "A", "B")}, names
    grab = choose(wm, ("grab", "obj1", "A"), 0)
    assert set(grab["add"]) == set(EXPECTED_GRAB["add"])
    assert set(grab["delete"]) == set(EXPECTED_GRAB["delete"])
    assert wm.snapshot() == before

acceptance("Grounded actions and read-only applicability", test_initial_applicability)

## 4. Required failure demonstration — effects only (§2.1.1)

**Predict:** if we only assert the grab's add set at time 1, what remains at time 1?

Expected: **only `holding(obj1, 1)`**. The hand's position, obj2's location, object declarations, locations and routes are absent at time 1. The time-0 facts may still be in the history, but they do not establish anything at time 1.

The following intentionally incomplete baseline is demonstration code. Do not mistake it for the correct `step` implementation. It uses the hand-written grab instance so you can record the frame problem before your engine is ready.

In [ ]:
effects_only_history = DEMO_FACTS | {
    effect + (1,) for effect in EXPECTED_GRAB["add"]
}
show_state(effects_only_history, 1)
assert at_time(effects_only_history, 1) == {("holding", "obj1", 1)}
assert ("at", "obj2", "C", 0) in effects_only_history
assert ("at", "obj2", "C", 1) not in effects_only_history
print("EXPECTED FAILURE RECORDED: untouched world facts are missing at t=1.")

def test_effects_only_stalls():
    wm = WorkingMemory(effects_only_history)
    assert not instances(wm, SCHEMAS["move"], 1)
    assert not instances(wm, SCHEMAS["release"], 1)

acceptance("Effects-only world cannot move or release at t=1", test_effects_only_stalls)

## 5. Persistence — predict the complete next state (§2.1.2)

Strip the current facts' time arguments to form `Sₜ`. For one grounded action:

**Sₜ₊₁ = (Sₜ − delete) ∪ add**, with all resulting facts stamped `t+1`.

This describes the required semantics; your team implements the generic engine loop. Old stamped facts stay in WM as history. Deleting `empty(hand)` means it is absent at time 1, not that `empty(hand, 0)` is erased.

**Predict:** after grabbing obj1, there are 10 facts at time 1. The hand is still at A, holding obj1; obj1 has no separate location, and obj2 remains at C. All declarations and routes survive.

In [ ]:
# Hand-written expected worlds: independent of the engine under test.
BACKGROUND = {
    ("object", "obj1"), ("object", "obj2"),
    ("location", "A"), ("location", "B"), ("location", "C"),
    ("route", "A", "B"), ("route", "B", "C"),
}
EXPECTED_CONTENTS = {
    0: BACKGROUND | {("at", "hand", "A"), ("at", "obj1", "A"),
                     ("at", "obj2", "C"), ("empty", "hand")},
    1: BACKGROUND | {("at", "hand", "A"), ("holding", "obj1"),
                     ("at", "obj2", "C")},
    2: BACKGROUND | {("at", "hand", "B"), ("holding", "obj1"),
                     ("at", "obj2", "C")},
    3: BACKGROUND | {("at", "hand", "B"), ("at", "obj1", "B"),
                     ("at", "obj2", "C"), ("empty", "hand")},
}
EXPECTED_STATES = {
    t: {fact + (t,) for fact in state} for t, state in EXPECTED_CONTENTS.items()
}
assert EXPECTED_STATES[0] == DEMO_FACTS
for t in range(4):
    print(f"Expected t={t}: {len(EXPECTED_STATES[t])} facts")

def test_one_step():
    wm = WorkingMemory(DEMO_FACTS)
    apply_instance(wm, choose(wm, ("grab", "obj1", "A"), 0), 0)
    show_state(wm.facts, 1)
    check_state(wm.facts, 1, EXPECTED_STATES[1])
    assert wm.snapshot() == EXPECTED_STATES[0] | EXPECTED_STATES[1]

acceptance("Grab with generic persistence and preserved history", test_one_step)

## 6. Required multi-step demonstration (§2.2)

Predict each row before running. This is a chosen sequence, not an automatically discovered plan.

| State | Action just completed | Hand | obj1 | obj2 | Hand status |
|---|---|---|---|---|---|
| 0 | Initial | A | A | C | Empty |
| 1 | Grab obj1 at A | A | Held | C | Holding obj1 |
| 2 | Move A → B | B | Held | C | Holding obj1 |
| 3 | Release obj1 at B | B | B | C | Empty |

**Target:** `at(obj1, B, 3)` while `at(obj2, C, 3)` still holds. Print the complete state after every action, as the assignment requests.

In [ ]:
SEQUENCE = [("grab", "obj1", "A"), ("move", "A", "B"),
            ("release", "obj1", "B")]

def test_sequence():
    wm = WorkingMemory(DEMO_FACTS)
    show_state(wm.facts, 0)
    for t, action_name in enumerate(SEQUENCE):
        old_history = wm.snapshot()
        action = choose(wm, action_name, t)
        print("\nFIRE:", action["name"], f"at t={t}")
        print("  Added at next step:", action["add"])
        print("  Not carried forward:", action["delete"])
        apply_instance(wm, action, t)
        show_state(wm.facts, t + 1)
        check_state(wm.facts, t + 1, EXPECTED_STATES[t + 1])
        assert old_history <= wm.facts, "History was erased"
        assert wm.snapshot() == set().union(*(EXPECTED_STATES[i] for i in range(t + 2)))
        assert ("at", "obj2", "C", t + 1) in wm.facts
    assert wm.holds(("at", "obj1", "B", 3))
    assert wm.holds(("at", "obj2", "C", 3))
    assert not wm.holds(("holding", "obj1", 3))
    print("TARGET REACHED; untouched object preserved at every step.")
    return wm

acceptance("Grab → move → release; exact states and target", test_sequence)

## 7. Check time isolation and unavailable actions

History must not make a past condition count as a present condition. At time 2, `empty(hand, 0)` exists, but `empty(hand, 2)` does not. The hand cannot grab another object while holding obj1. After release at time 3, it cannot release obj1 again.

The minimum contract here is that invalid instances are **not yielded by `applicable`**. Whether `step` also rejects an invalid or stale instance is a team API choice; if you promise rejection, add a test that it leaves WM unchanged on failure.

In [ ]:
def test_time_isolation():
    wm = WorkingMemory(DEMO_FACTS)
    apply_instance(wm, choose(wm, SEQUENCE[0], 0), 0)
    assert not instances(wm, SCHEMAS["grab"], 1)
    apply_instance(wm, choose(wm, SEQUENCE[1], 1), 1)
    assert not instances(wm, SCHEMAS["grab"], 2)
    assert {tuple(a["name"]) for a in instances(wm, SCHEMAS["move"], 2)} == {
        ("move", "B", "C")
    }, "Historical A must not enable another A → B move"
    apply_instance(wm, choose(wm, SEQUENCE[2], 2), 2)
    assert not instances(wm, SCHEMAS["release"], 3)
    assert list(wm.ask(("at", "obj1", "?where", 0))) == [{"?where": "A"}]
    assert list(wm.ask(("at", "obj1", "?where", 3))) == [{"?where": "B"}]
    assert list(wm.ask(("at", "obj2", "?where", 1))) == [{"?where": "C"}]

acceptance("Historical facts do not satisfy current preconditions", test_time_isolation)

## 8. Variables and multiple objects — same schemas, different world

Reset to a fresh scenario where the hand and **both objects** start at C. Expect two grab instances: one for obj1 and one for obj2. Select obj2. The next state must contain `holding(obj2, 1)` and preserve `at(obj1, C, 1)`.

This catches an implementation that always chooses obj1 or stops after the first condition match. No engine changes are allowed for this scenario.

In [ ]:
def test_multiple_objects():
    facts = (DEMO_FACTS - {("at", "hand", "A", 0), ("at", "obj1", "A", 0)}) | {
        ("at", "hand", "C", 0), ("at", "obj1", "C", 0)
    }
    wm = WorkingMemory(facts)
    options = instances(wm, SCHEMAS["grab"], 0)
    assert {tuple(a["name"]) for a in options} == {
        ("grab", "obj1", "C"), ("grab", "obj2", "C")
    }
    selected = next(a for a in options if tuple(a["name"]) == ("grab", "obj2", "C"))
    apply_instance(wm, selected, 0)
    assert wm.holds(("holding", "obj2", 1))
    assert wm.holds(("at", "obj1", "C", 1))
    assert not wm.holds(("at", "obj2", "C", 1))
    assert not wm.holds(("empty", "hand", 1))

acceptance("All bindings and either object supported", test_multiple_objects)

## 9. Engine/knowledge separation — unfamiliar facts and a new action

A persistence engine should carry any timed fact, not just known predicates. Add `inspection_tag(obj2, blue, 0)`: it must survive the grab. Then add a **new action dictionary** that changes that tag, and confirm that the generic engine handles it without any code change.

This small extension tests both generic persistence and generic deletion. It is an acceptance check, not an extra assignment feature you need to present.

In [ ]:
RETAG_ACTION = {
    "name": ("retag", "?o"),
    "if": [("object", "?o", "?t"), ("inspection_tag", "?o", "blue", "?t")],
    "add": [("inspection_tag", "?o", "green")],
    "delete": [("inspection_tag", "?o", "blue")],
}

def test_generic_data():
    wm = WorkingMemory(DEMO_FACTS | {("inspection_tag", "obj2", "blue", 0)})
    apply_instance(wm, choose(wm, ("grab", "obj1", "A"), 0), 0)
    assert wm.holds(("inspection_tag", "obj2", "blue", 1))
    options = instances(wm, RETAG_ACTION, 1)
    assert len(options) == 1
    assert tuple(options[0]["name"]) == ("retag", "obj2")
    apply_instance(wm, options[0], 1)
    expected = (EXPECTED_CONTENTS[1] | {("inspection_tag", "obj2", "green")})
    assert contents(wm.facts, 2) == expected
    assert wm.holds(("inspection_tag", "obj2", "blue", 1))
    assert not wm.holds(("inspection_tag", "obj2", "blue", 2))

acceptance("Unfamiliar predicate and new action handled as data", test_generic_data)

## 10. Milestones you can work on while Task 1 continues

| Milestone | Runnable evidence | Work possible before the temporal engine exists |
|---|---|---|
| A. Representation agreed | Cells 1–5 run; everyone predicts the same states | Review domain data, sketch states, document action contract |
| B. Applicability | Section 3 passes | Reuse Task 1's condition matcher; first bind `?t` |
| C. Failure recorded | Section 4 displays the one-fact state | Capture explanation and expected output for the report |
| D. One correct transition | Section 5 passes | Compare exact expected next state and preserved history |
| E. Full sequence | Sections 6–7 pass | Prepare target, invalid-action and history checks |
| F. Generic solution | Sections 8–9 pass | Review engine for domain words; move fixtures into `domains.py` |

Do not run Task 1's unmodified forward chainer over the actions: they have add/delete effects and advance time, rather than a single monotonic `then` conclusion. Reuse generic matching machinery and WM; implement the temporal transition separately.

Keep presentation/display code and these checks in the notebook, engine code in `engine.py` (or its generic modules), and knowledge data in `domains.py`. Once domain data is moved, import it here rather than maintaining divergent copies.

## 11. Short final presentation sequence

Allow roughly 3 minutes for Task 2 within the joint 7–10 minute presentation:

1. Show the initial world and the action format (about 30 seconds).
2. Show the effects-only time-1 state; name the facts that disappeared (about 30 seconds).
3. Run the three-action sequence with persistence; show each state (about 60 seconds).
4. Check the target, unchanged obj2, and one historical query (about 30 seconds).
5. Explain the frame assumption and the distinction from Task 1 (about 30 seconds).

Keep the longer acceptance checks for development and questions. Prepare saved output as a presentation backup.

## 12. Report prompts and completion checklist

- [ ] Explain why facts include time and effects omit it until stamped by the engine.
- [ ] Record both the effects-only failure and the persistence result.
- [ ] Explain how unchanged facts survive without listing them in each action.
- [ ] Explain why writing frame statements for each fact/action combination grows poorly as the world and action set expand.
- [ ] Explain that Task 1 infers relationships at one world state; Task 2 models causal transitions between states.
- [ ] Explain that history may grow monotonically in storage even though the **current world state** changes and propositions stop holding at later times.
- [ ] Describe one representation tried and rejected, based on what your team actually tried (do not invent an experiment).
- [ ] Verify variables, multiple objects, time isolation, history and engine/knowledge separation.
- [ ] Enable checks, restart the kernel and Run All successfully on another team member's machine.

**Boundary:** this scaffold specifies expected behaviour and exercises your implementation. It does not supply the required `applicable` or `step` algorithms, automatic planning, or a completed submission.